In [1]:
import pyodbc
import pandas as pd
from pycaret.classification import setup, compare_models
import numpy as np


In [2]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"        # default instance
    "DATABASE=LOAN_PORTFOLIO_DB;"
    "Trusted_Connection=yes"   # use Windows Authentication
)

In [3]:
customer_df = pd.read_sql("SELECT * FROM CUSTOMER_DETAILS", conn)
account_df = pd.read_sql("SELECT * FROM ACCOUNT_DETAILS", conn)
transaction_df = pd.read_sql("SELECT * FROM TRANSACTION_DETAILS", conn)
loan_cashflow_df = pd.read_sql("SELECT * FROM LOAN_CASHFLOW", conn)
repayment_df = pd.read_sql("SELECT * FROM REPAYMENT", conn)

In [4]:
customer_df['MARITAL_STATUS'].fillna('Unknown', inplace=True)
customer_df['EMPLOYMENT_STATUS'].fillna('Unknown', inplace=True)
customer_df['DISTRICT'].fillna('Unknown', inplace=True)
customer_df['OCCUPATION'].fillna('Unknown', inplace=True)
median_risk = customer_df['CUSTOMER_RISK'].median()
customer_df['CUSTOMER_RISK'].fillna(median_risk, inplace=True)


In [5]:
accts_numeric_cols = ['TERM', 'TERM_AMOUNT', 'OOD', 'MONTHEND_CONVERTED_BALANCE', 'CONVERTED_BALANCE',
                'JUN_25','JUL_25','AUG_25','SEP_25','OCT_25','NOV_25']

account_df[accts_numeric_cols] = account_df[accts_numeric_cols].fillna(0)

account_df['ACCT_CLOSE_DATE'] = pd.to_datetime(account_df['ACCT_CLOSE_DATE'], errors='coerce')


In [6]:
loan_numeric_cols = ['OOD', 'CAPITAL_TO_BE_PAIED', 'INTEREST_TO_BE_PAIED']
loan_cashflow_df[loan_numeric_cols] = loan_cashflow_df[loan_numeric_cols].fillna(0)

loan_cashflow_df['TO_BE_PAYMENT_DATE'] = pd.to_datetime(loan_cashflow_df['TO_BE_PAYMENT_DATE'], errors='coerce')


In [7]:
repayment_numeric_cols = ['OOD', 'CAPITAL_PAIED']
repayment_df[repayment_numeric_cols] = repayment_df[repayment_numeric_cols].fillna(0)



In [8]:
# Rule 1 - Hard Eligibility & Regulatory Rules

In [9]:
## Regulatory Age Check

In [10]:
customer_df['Eligibility_Flag'] = 'ELIGIBLE'
customer_df['Rejection_Reason'] = None

In [11]:
customer_df.loc[
    (customer_df['AGE'] < 18) | (customer_df['AGE'] > 80),
    ['Eligibility_Flag', 'Rejection_Reason']
] = ['REJECT', 'Regulatory Age Restriction']


In [12]:
eligible_cus_df = customer_df[customer_df['Eligibility_Flag'] == 'ELIGIBLE'].copy()


In [13]:
## Account Eligibility Check

In [14]:
active_accounts = account_df[
    (account_df['ACCT_STATUS'] == 'ACTIVE') |
    (account_df['ACCT_CLOSE_DATE'].isna())
]

active_account_count = (
    active_accounts
    .groupby('MASKED_ID')
    .size()
    .reset_index(name='Number_of_Active_Accounts')
)

In [15]:
eligible_cus_df = eligible_cus_df.merge(
    active_account_count,
    on='MASKED_ID',
    how='left'
)

In [16]:

# if no accounts isnted of Nan use 0 
eligible_cus_df['Number_of_Active_Accounts'] = (
    eligible_cus_df['Number_of_Active_Accounts']
    .fillna(0)
    .astype(int)
)


In [17]:
# Step 1: Set ELSE condition (default)
eligible_cus_df['Eligibility_Flag'] = 'ELIGIBLE'
eligible_cus_df['Rejection_Reason'] = 'Existing Customer'

# Step 2: Apply IF condition
eligible_cus_df.loc[
    eligible_cus_df['Number_of_Active_Accounts'] == 0,
    ['Eligibility_Flag', 'Rejection_Reason']
] = ['REJECT', 'Non-Existing Customer']

In [18]:
## Employment Status Validation

In [19]:
valid_employment_status = pd.read_sql(
    "SELECT DISTINCT EMPLOYMENT_STATUS FROM CUSTOMER_DETAILS",
    conn
)['EMPLOYMENT_STATUS'].dropna().tolist()

In [20]:
eligible_cus_df['Employment_Status_Flag'] = 'Valid Employment Status'

eligible_cus_df.loc[
    ~eligible_cus_df['EMPLOYMENT_STATUS'].isin(valid_employment_status),
    'Employment_Status_Flag'
] = 'Invalid Employment Status'

In [21]:
# Rule 2 - Employment-Based Routing & Participation Rules

In [22]:
## Routing & Participation 

In [23]:
eligible_cus_df['Employment_Segment'] = 'Other'

In [24]:
eligible_cus_df.loc[
    (eligible_cus_df['EMPLOYMENT_STATUS'].isin(['EMPLOYED', 'SELF-EMPLOYED', 'BUSINESS'])) &
    (eligible_cus_df['AGE'].between(18, 60)),
    'Employment_Segment'
] = 'Core Working Group'


In [25]:
eligible_cus_df.loc[
    (eligible_cus_df['EMPLOYMENT_STATUS'].isin(['UNEMPLOYED	', 'RETIRED', 'STUDENT', 'FREELANCE'])) &
    (eligible_cus_df['AGE'].between(18, 65)),
    'Employment_Segment'
] = 'Special Segment'

In [26]:
eligible_cus_df.loc[
    (eligible_cus_df['EMPLOYMENT_STATUS'].isin([
        'UNEMPLOYED', 'RETIRED', 'STUDENT', 'FREELANCE', 
        'EMPLOYED', 'SELF-EMPLOYED', 'BUSINESS'
    ])) &
    (~eligible_cus_df['AGE'].between(18, 60)),  # not between 18 and 60
    'Employment_Segment'
] = 'Not valid segment'

In [27]:
# Rule 3 - Age-Based Segments

In [28]:
eligible_cus_df['Age_Bucket'] = pd.cut(
    eligible_cus_df['AGE'],
    bins=[17, 25, 40, 60, 80],
    labels=['Young Adult', 'Adult', 'Middle-Aged', 'Senior']
)

In [29]:
# Rule 4 - Financial capacity

In [30]:
balance_cols = ['JUN_25', 'JUL_25', 'AUG_25', 'SEP_25', 'OCT_25', 'NOV_25']

In [31]:
account_df['Monthly_Avg_Balance'] = account_df[balance_cols].mean(axis=1)

In [32]:
customer_balance_df = (
    account_df
    .groupby('MASKED_ID', as_index=False)['Monthly_Avg_Balance']
    .mean()
)


In [33]:
eligible_cus_df = eligible_cus_df.merge(
    customer_balance_df,
    on='MASKED_ID',
    how='left'
)


In [34]:
eligible_cus_df['Financial_Capacity'] = 'Unknown / Missing Balance Data'

eligible_cus_df.loc[
    eligible_cus_df['Monthly_Avg_Balance'] >= 100000,
    'Financial_Capacity'
] = 'High Financial Capacity'

eligible_cus_df.loc[
    eligible_cus_df['Monthly_Avg_Balance'].between(50000, 99999),
    'Financial_Capacity'
] = 'Medium Financial Capacity'

eligible_cus_df.loc[
    eligible_cus_df['Monthly_Avg_Balance'] < 50000,
    'Financial_Capacity'
] = 'Low Financial Capacity'


In [35]:
# Rule 6 - Salary Verification Check

In [36]:
transaction_df['AMOUNT_LCY'] = pd.to_numeric(
    transaction_df['AMOUNT_LCY'],
    errors='coerce'
)

In [37]:
credit_df = transaction_df[
    transaction_df['AMOUNT_LCY'] > 0
].copy()


In [38]:
credit_df['Month'] = pd.to_datetime(
    credit_df['BOOKING_DATE']
).dt.to_period('M')

In [39]:
monthly_credit = (
    credit_df
    .groupby(['MASKED_ID', 'Month'], as_index=False)['AMOUNT_LCY']
    .sum()
)

In [40]:
avg_monthly_income = (
    monthly_credit
    .groupby('MASKED_ID', as_index=False)['AMOUNT_LCY']
    .mean()
    .rename(columns={'AMOUNT_LCY': 'Avg_Monthly_Credit'})
)

In [41]:
eligible_cus_df['MASKED_ID'] = eligible_cus_df['MASKED_ID'].astype(str)


In [42]:
avg_monthly_income['MASKED_ID'] = avg_monthly_income['MASKED_ID'].astype(str)

In [43]:
eligible_cus_df = eligible_cus_df.merge(
    avg_monthly_income,
    on='MASKED_ID',
    how='left'
)


In [44]:
mask = (
    eligible_cus_df['Avg_Monthly_Credit'].isna() &
    (eligible_cus_df['Monthly_Avg_Balance'] > 0)
)

eligible_cus_df.loc[mask, 'Avg_Monthly_Credit'] = (
    eligible_cus_df.loc[mask, 'Monthly_Avg_Balance']
)

In [45]:

eligible_cus_df['Cluster_Name'] = 'Unknown / Missing Salary'

eligible_cus_df.loc[
    eligible_cus_df['Avg_Monthly_Credit'] >= 100000,
    'Cluster_Name'
] = 'High Salary'

eligible_cus_df.loc[
    eligible_cus_df['Avg_Monthly_Credit'].between(50000, 99999),
    'Cluster_Name'
] = 'Medium Salary'

eligible_cus_df.loc[
    eligible_cus_df['Avg_Monthly_Credit'] < 50000,
    'Cluster_Name'  
] = 'Low Salary'


In [46]:
#Creating Credit scores

In [47]:

# # Ensure AMOUNT_LCY is numeric

# transaction_df["AMOUNT_LCY"] = pd.to_numeric(transaction_df["AMOUNT_LCY"], errors='coerce').fillna(0)


# Create inflow, outflow, net, and absolute activity

transaction_df["INFLOW"] = transaction_df["AMOUNT_LCY"].apply(lambda x: x if x > 0 else 0)
transaction_df["OUTFLOW"] = transaction_df["AMOUNT_LCY"].apply(lambda x: abs(x) if x < 0 else 0)
transaction_df["NET_AMOUNT"] = transaction_df["AMOUNT_LCY"]  # keeps signs
transaction_df["ABS_ACTIVITY"] = transaction_df["AMOUNT_LCY"].abs()


#  Aggregate per customer

txn_features = transaction_df.groupby("MASKED_ID").agg({
    "INFLOW": "sum",
    "OUTFLOW": "sum",
    "NET_AMOUNT": "sum",
    "ABS_ACTIVITY": "sum",
    "AMOUNT_LCY": ["mean", "std", "count"]  # original avg/std/count
}).reset_index()


# Flatten MultiIndex columns from aggregation

txn_features.columns = [
    "MASKED_ID",
    "TOTAL_INFLOW",
    "TOTAL_OUTFLOW",
    "NET_TRANSACTION_SUM",
    "ABS_ACTIVITY",
    "AVG_TRANSACTION",
    "STD_TRANSACTION",
    "TXN_COUNT"
]

txn_features["INFLOW_RATIO"] = txn_features["TOTAL_INFLOW"] / (txn_features["ABS_ACTIVITY"] + 1)
txn_features["OUTFLOW_RATIO"] = txn_features["TOTAL_OUTFLOW"] / (txn_features["ABS_ACTIVITY"] + 1)
txn_features["NET_RATIO"] = txn_features["NET_TRANSACTION_SUM"] / (txn_features["ABS_ACTIVITY"] + 1)


# Average transaction per day (assuming 30 days)
txn_features["AVG_TXN_PER_DAY"] = txn_features["TXN_COUNT"] / 30

# Transaction volatility: std relative to average
txn_features["TRANSACTION_VOLATILITY"] = txn_features["STD_TRANSACTION"] / (txn_features["AVG_TRANSACTION"] + 1)


# Average monthly inflow as income proxy
txn_features["AVG_MONTHLY_INFLOW"] = txn_features["TOTAL_INFLOW"] / 12

# Log-transformed average inflow for modeling stability
txn_features["LOG_AVG_INFLOW"] = np.log1p(txn_features["AVG_MONTHLY_INFLOW"])



In [48]:

# Fill missing values with 0

num_cols = txn_features.select_dtypes(include=["float64", "int64"]).columns
txn_features[num_cols] = txn_features[num_cols].fillna(0)


In [ ]:
# ── Step 1: Clean loan_cashflow_df numeric columns ──
for col in ["CAPITAL_TO_BE_PAIED", "INTEREST_TO_BE_PAIED", "TERM_AMOUNT"]:
    loan_cashflow_df[col] = pd.to_numeric(
        loan_cashflow_df[col].astype(str).str.replace(",", "").str.replace(" ", ""),
        errors='coerce'
    ).fillna(0)

# ── Step 2: Clean repayment_df numeric columns ──
for col in ["CAPITAL_PAIED", "INTEREST_PAIED"]:
    repayment_df[col] = pd.to_numeric(
        repayment_df[col].astype(str).str.replace(",", "").str.replace(" ", ""),
        errors='coerce'
    ).fillna(0)

# ── Step 3: Total scheduled amounts per customer ──
scheduled = loan_cashflow_df.groupby("MASKED_ID").agg(
    TOTAL_CAPITAL_SCHEDULED  = ("CAPITAL_TO_BE_PAIED", "sum"),
    TOTAL_INTEREST_SCHEDULED = ("INTEREST_TO_BE_PAIED", "sum"),
    TOTAL_AMOUNT_SCHEDULED   = ("TERM_AMOUNT",          "sum")
).reset_index()

# ── Step 4: Total actually paid per customer ──
actually_paid = repayment_df.groupby("MASKED_ID").agg(
    TOTAL_CAPITAL_PAID  = ("CAPITAL_PAIED",  "sum"),
    TOTAL_INTEREST_PAID = ("INTEREST_PAIED", "sum")
).reset_index()

# ── Step 5: Payment behaviour — how consistently do they pay ──
payment_behaviour = repayment_df.groupby("MASKED_ID").agg(
    MONTHS_WITH_PAYMENT = ("CAPITAL_PAIED", lambda x: (x > 0).sum()),
    TOTAL_MONTHS        = ("CAPITAL_PAIED", "count")
).reset_index()

payment_behaviour["PAYMENT_FREQUENCY"] = (
    payment_behaviour["MONTHS_WITH_PAYMENT"] /
    payment_behaviour["TOTAL_MONTHS"].replace(0, 1)
)

# ── Step 6: Merge everything together ──
cash_features = scheduled \
    .merge(actually_paid,      on="MASKED_ID", how="left") \
    .merge(payment_behaviour,  on="MASKED_ID", how="left")

# ── Step 7: Force numeric on all columns ──
numeric_cols = [
    "TOTAL_CAPITAL_PAID",
    "TOTAL_INTEREST_PAID",
    "TOTAL_CAPITAL_SCHEDULED",
    "TOTAL_INTEREST_SCHEDULED",
    "TOTAL_AMOUNT_SCHEDULED",
    "MONTHS_WITH_PAYMENT",
    "TOTAL_MONTHS",
    "PAYMENT_FREQUENCY"
]
for col in numeric_cols:
    cash_features[col] = pd.to_numeric(
        cash_features[col], errors='coerce'
    ).fillna(0)

# ── Step 8: Calculate DUE columns (for DEFAULT definition only) ──
cash_features["TOTAL_CAPITAL_DUE"] = (
    cash_features["TOTAL_CAPITAL_SCHEDULED"] - cash_features["TOTAL_CAPITAL_PAID"]
)
cash_features["AVG_PAYMENT_RATIO"] = (
    cash_features["TOTAL_CAPITAL_PAID"] /
    cash_features["TOTAL_AMOUNT_SCHEDULED"].replace(0, 1)
)

# ── Step 9: Keep final columns ──
cash_features = cash_features[[
    "MASKED_ID",

    # ── For DEFAULT definition only — do NOT put in features ──
    "TOTAL_CAPITAL_DUE",       # used in DEFAULT condition 3
    "AVG_PAYMENT_RATIO",       # used in DEFAULT condition 3

    # ── Safe to use as model features ──
    "TOTAL_CAPITAL_SCHEDULED", # how much they borrowed
    "TOTAL_INTEREST_SCHEDULED",# cost of the loan
    "TOTAL_AMOUNT_SCHEDULED",  # total loan obligation
    "PAYMENT_FREQUENCY",       # how consistently they pay — key behavioural signal
    "MONTHS_WITH_PAYMENT",     # total months they made a payment
    "TOTAL_MONTHS",            # total months on record
]].fillna(0)






In [50]:
import numpy as np

# Ensure numeric columns
account_df["MONTHEND_CONVERTED_BALANCE"] = pd.to_numeric(account_df["MONTHEND_CONVERTED_BALANCE"], errors='coerce').fillna(0)
account_df["CONVERTED_BALANCE"] = pd.to_numeric(account_df["CONVERTED_BALANCE"], errors='coerce').fillna(0)
account_df["OOD"] = pd.to_numeric(account_df["OOD"], errors='coerce').fillna(0)

# Calculate Utilization: ratio of loan balance / term amount
# (if TERM_AMOUNT available, else skip or adjust)
account_df["UTILIZATION"] = account_df["CONVERTED_BALANCE"] / account_df["TERM_AMOUNT"].replace(0, 1)

# Calculate ACCOUNT_AGE_MONTHS
# ----------------------------
# Ensure date columns are datetime
account_df["ORIG_CONTRACT_DATE"] = pd.to_datetime(account_df["ORIG_CONTRACT_DATE"], errors='coerce')
account_df["ACCT_CLOSE_DATE"] = pd.to_datetime(account_df["ACCT_CLOSE_DATE"], errors='coerce')

# Fix reference date for reproducibility (e.g., model training cutoff)
reference_date = pd.to_datetime("2026-01-31")

# Use ACCT_CLOSE_DATE if exists, else reference_date
account_df["END_DATE"] = account_df["ACCT_CLOSE_DATE"].fillna(reference_date)

# Calculate account age in months
account_df["ACCOUNT_AGE_MONTHS"] = ((account_df["END_DATE"] - account_df["ORIG_CONTRACT_DATE"]) / np.timedelta64(1, "M")).round(0)

# Aggregate per customer if multiple accounts
acc_features = account_df.groupby("MASKED_ID").agg({
    "MONTHEND_CONVERTED_BALANCE": "mean",
    "UTILIZATION": "mean",
    # "OOD": "max",
    "ACCOUNT_AGE_MONTHS": "mean"
}).reset_index()

# Fill missing numeric values
acc_features = acc_features.fillna(0)


In [51]:
repayment_df["OOD"] = pd.to_numeric(
    repayment_df["OOD"],
    errors="coerce"
)

In [52]:
re_numeric_cols = ["CAPITAL_PAIED", "INTEREST_PAIED"]

for col in re_numeric_cols:
    repayment_df[col] = pd.to_numeric(repayment_df[col], errors="coerce")  # convert strings to numbers
    repayment_df[col].fillna(0, inplace=True)  # replace NaN with 0

In [53]:
ood_features = repayment_df.groupby("MASKED_ID").agg({
    "OOD": "max"
}).reset_index()

ood_features.rename(columns={"OOD": "MAX_OOD"}, inplace=True)


In [54]:
payment_features = repayment_df.groupby("MASKED_ID").agg({
    "CAPITAL_PAIED": "sum",
    "INTEREST_PAIED": "sum"
}).reset_index()

payment_features["TOTAL_PAID"] = (
    payment_features["CAPITAL_PAIED"] +
    payment_features["INTEREST_PAIED"]
)

In [55]:
model_data = eligible_cus_df.copy()

In [56]:
model_data = model_data.merge(txn_features, on="MASKED_ID", how="left")

In [57]:
model_data = model_data.merge(cash_features, on="MASKED_ID", how="left")

In [58]:
model_data = model_data.merge(acc_features, on="MASKED_ID", how="left")

In [59]:
# Merge customer-level cash features with MAX_OOD
model_data = model_data.merge(ood_features, on="MASKED_ID", how="left")

# Fill missing MAX_OOD for customers with no loans
model_data["MAX_OOD"] = model_data["MAX_OOD"].fillna(0)
#

In [60]:
model_data = model_data.merge(payment_features, on="MASKED_ID", how="left")

# Fill missing MAX_OOD for customers with no loans
model_data["TOTAL_PAID"] = model_data["TOTAL_PAID"].fillna(0)

In [61]:
num_cols = model_data.select_dtypes(include=["float64", "int64"]).columns
model_data[num_cols] = model_data[num_cols].fillna(0)

In [62]:
# Identify categorical columns
cat_cols = model_data.select_dtypes(include=["category", "object"]).columns


In [63]:

for col in cat_cols:
    if pd.api.types.is_categorical_dtype(model_data[col]):
        # Add 'Unknown' to categories if not already present
        if "Unknown" not in model_data[col].cat.categories:
            model_data[col] = model_data[col].cat.add_categories("Unknown")
    # Fill missing values
    model_data[col] = model_data[col].fillna("Unknown")


In [64]:
# Creating the default flag

In [ ]:

#defining the defualt 
loan_status = account_df[
    account_df["ACTIVE_PRODUCT"].isin(["LOAN ACCOUNT", "BORROWINGS"])
].groupby("MASKED_ID").agg(
    MAX_OOD_RAW  = ("OOD", "max"),
    WORST_STATUS = ("ACCT_STATUS", lambda x: 
                   "BAD" if any(s in ["ABANDONED", "UNCLAIMED", "EXPIRED"] 
                   for s in x) else "OK")
).reset_index()

# Merge into model_data
if "MAX_OOD_RAW" not in model_data.columns:
    model_data = model_data.merge(loan_status, on="MASKED_ID", how="left")
else:
    model_data = model_data.drop(columns=["MAX_OOD_RAW"], errors="ignore")
    model_data = model_data.merge(loan_status, on="MASKED_ID", how="left")

model_data["MAX_OOD_RAW"]  = model_data["MAX_OOD_RAW"].fillna(0)
model_data["WORST_STATUS"] = model_data["WORST_STATUS"].fillna("OK")

# ── Combined DEFAULT definition ──
model_data["DEFAULT"] = (
    # Condition 1: 30+ days overdue
    (model_data["MAX_OOD_RAW"] >= 30) |

    # Condition 2: Bad account status
    (model_data["WORST_STATUS"] == "BAD") |

    # Condition 3: Has capital due AND very poor payment ratio
    # (use TOTAL_CAPITAL_DUE and AVG_PAYMENT_RATIO from account_df
    #  NOT from model features — separate calculation!)
    (
        (model_data["TOTAL_CAPITAL_DUE"] > 0) &
        (model_data["AVG_PAYMENT_RATIO"] < 0.3)
    )
).astype(int)


In [ ]:
features = [
    # Transaction behaviour — ✅ all clean, no link to DEFAULT
    "NET_TRANSACTION_SUM",
    "AVG_TRANSACTION",
    "STD_TRANSACTION",
    "TXN_COUNT",
    "NET_RATIO",
    "AVG_TXN_PER_DAY",
    "TRANSACTION_VOLATILITY",
    "LOG_AVG_INFLOW",

    # Loan size — ✅ scheduled amounts, exist before default happens
    "TOTAL_CAPITAL_SCHEDULED",    # how much they borrowed
    "TOTAL_INTEREST_SCHEDULED",   # cost of their loan
 

    # Payment behaviour — ✅ pattern based, not consequence of default
    "PAYMENT_FREQUENCY",          # how consistently they pay (0.0 to 1.0)
    


    # Account level — ✅ all clean
    "MONTHEND_CONVERTED_BALANCE",
    "UTILIZATION",
    "ACCOUNT_AGE_MONTHS",
    "Number_of_Active_Accounts",

    # Risk
    "CUSTOMER_RISK_NAME",
]

# Demographic features
demographic_features = [
    "AGE",
    "EMPLOYMENT_STATUS",
    "OCCUPATION",
    "MARITAL_STATUS",
    "DISTRICT",
]

# Combine
features = features + demographic_features

# Categorical columns
categorical_cols = [
    "EMPLOYMENT_STATUS",
    "OCCUPATION",
    "MARITAL_STATUS",
    "DISTRICT",
    "CUSTOMER_RISK_NAME",
    # "GENDER"         ❌ removed
]

for col in categorical_cols:
    model_data[col] = model_data[col].astype("category")

print(f"Total features: {len(features)}")
print(features)

In [68]:
X = model_data[features]
y = model_data["DEFAULT"]

In [ ]:
#xgboost monotone constrain 

In [70]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# ── Step 1: Prepare full dataset with category dtype ──
X_full = X.copy()
for col in categorical_cols:
    X_full[col] = X_full[col].astype("category")

# ── Step 2: Rebuild monotone constraints from X_full columns ──
# (X_full has same columns as X_train but we rebuild to be safe)
constraint_map = {
      # Transaction behaviour
    "NET_TRANSACTION_SUM"        : -1,  # more net flow       → lower risk
    "AVG_TRANSACTION"            : -1,  # higher avg txn      → lower risk
    "STD_TRANSACTION"            :  1,  # high volatility     → higher risk
    "TXN_COUNT"                  : -1,  # more transactions   → lower risk
    "NET_RATIO"                  : -1,  # better ratio        → lower risk
    "AVG_TXN_PER_DAY"            : -1,  # more activity       → lower risk
    "TRANSACTION_VOLATILITY"     :  1,  # more volatile       → higher risk
    "LOG_AVG_INFLOW"             : -1,  # more inflow         → lower risk

    # Loan size
    "TOTAL_CAPITAL_SCHEDULED"    :  1,  # bigger loan         → higher risk
    "TOTAL_INTEREST_SCHEDULED"   :  1,  # higher interest     → higher risk

    # Payment behaviour
    "PAYMENT_FREQUENCY"          : -1,  # pays more often     → lower risk


    # Account level
    "MONTHEND_CONVERTED_BALANCE" : -1,  # higher balance      → lower risk
    "UTILIZATION"                :  1,  # higher utilization  → higher risk
    "ACCOUNT_AGE_MONTHS"         : -1,  # older account       → lower risk
    "Number_of_Active_Accounts"  :  0,  # unclear direction

    # Categoricals — cannot be constrained
    "CUSTOMER_RISK_NAME"         :  0,
    "AGE"                        :  0,
    "EMPLOYMENT_STATUS"          :  0,
    "OCCUPATION"                 :  0,
    "MARITAL_STATUS"             :  0,
    "DISTRICT"                   :  0,

}

# Build from X_full columns — guarantees correct order
monotone_constraints_full = tuple(
    constraint_map.get(f, 0) for f in X_full.columns
)


# ── Step 3: Retrain on full dataset with same rules ──
non_default  = (y == 0).sum()
default      = (y == 1).sum()
weight_full  = non_default / default

xgb_monotone_full = xgb.XGBClassifier(
    # ── Exact same hyperparameters as tested model ──
    n_estimators         = 300,
    max_depth            = 4,
    learning_rate        = 0.05,
    subsample            = 0.8,
    colsample_bytree     = 0.8,
    reg_alpha            = 1,
    reg_lambda           = 2,
    min_child_weight     = 5,
    scale_pos_weight     = weight_full,
    random_state         = 42,
    eval_metric          = "logloss",
    enable_categorical   = True,
    # ── Same business rules enforced ──
    monotone_constraints = monotone_constraints_full
)





XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [ ]:


# ── Step 4: Predict ──
mon_y_proba_all = xgb_monotone_full.predict_proba(X_full)[:, 1]
mon_y_pred_all  = xgb_monotone_full.predict(X_full)

# ── Step 5: Build results ──
results = pd.DataFrame({
    "MASKED_ID"           : model_data["MASKED_ID"],
    "Default_Probability" : mon_y_proba_all,
    "Predicted_Default"   : mon_y_pred_all
})

# ── Step 6: Generate credit score (250 to 900) ──
results["Internal_Bank_Default_Score"] = (
    900 - (mon_y_proba_all * (900 - 250))
).round().astype(int)



In [ ]:
# ── Step 7: Score bands ──
bins   = [250, 500, 620, 750, 900]
labels = ["High Risk", "Medium Risk", "Low Risk", "Very Low Risk"]

results["Score_Band"] = pd.cut(
    results["Internal_Bank_Default_Score"],
    bins           = bins,
    labels         = labels,
    include_lowest = True
)

print("\nScore Band Distribution:")
print(results["Score_Band"].value_counts())
print(f"\nAverage credit score : {results['Internal_Bank_Default_Score'].mean():.0f}")
print(f"Score range          : {results['Internal_Bank_Default_Score'].min()} — {results['Internal_Bank_Default_Score'].max()}")


In [78]:
final_table = model_data.merge(results, on="MASKED_ID", how="left")

In [79]:
## Model 6 K-Prototypes

In [80]:
!pip install kmodes

Defaulting to user installation because normal site-packages is not writeable


In [81]:
import pandas as pd
from kmodes.kprototypes import KPrototypes

In [82]:
from kmodes.kprototypes import KPrototypes
import pandas as pd

# Use a separate copy for final clustering
eligible_final = final_table.copy()

In [83]:
# Define features
numeric_feats_final = ['Monthly_Avg_Balance', 'Avg_Monthly_Credit','AGE']
categorical_feats_final = ['OCCUPATION','CUSTOMER_RISK_NAME','GENDER', 
                           'EMPLOYMENT_STATUS', 'MARITAL_STATUS','TARGET_DESC','Score_Band']

In [84]:
# Fill missing values
eligible_final[numeric_feats_final] = eligible_final[numeric_feats_final].fillna(0)
for col in categorical_feats_final:
    eligible_final[col] = eligible_final[col].astype(str).fillna('Unknown')

In [85]:
# Prepare cluster data
cluster_data_final = eligible_final[numeric_feats_final + categorical_feats_final].copy()
# Weight Internal_Bank_Default_Score higher
weight_factor = 5  # you can tune this
cluster_data_final['Internal_Bank_Default_Score'] = eligible_final['Internal_Bank_Default_Score'] * weight_factor

In [86]:
eligible_final[categorical_feats_final] = eligible_final[categorical_feats_final].fillna('Unknown').astype(str)

In [87]:

cat_idx_final = [cluster_data_final.columns.get_loc(col) for col in categorical_feats_final]


In [88]:
# Fit K-Prototypes with k = 4
kproto_final = KPrototypes(n_clusters=4, init='Cao', random_state=42)
cluster_labels_final = kproto_final.fit_predict(cluster_data_final.values, categorical=cat_idx_final)

In [89]:
import joblib

# ----- SAVE THE CLUSTER MODEL -----
joblib.dump(kproto_final, "kproto_cluster_model.pkl")
print("K-Prototypes cluster model saved as kproto_cluster_model.pkl")

K-Prototypes cluster model saved as kproto_cluster_model.pkl


In [90]:
# Assign cluster labels to dataframe
eligible_final['Cluster_KProto'] = cluster_labels_final

In [91]:
# Merging to the original df

In [92]:
# Keep only MASKED_ID and NAME_MASKED_ID from customer_df
customer_basic = customer_df[['MASKED_ID']].copy()

# Drop duplicate MASKED_IDs in eligible_final first
eligible_final = eligible_final.drop_duplicates(subset='MASKED_ID')

# Perform left join on MASKED_ID
customer_full_df = customer_basic.merge(
    eligible_final,
    on='MASKED_ID',
    how='left'  # keeps all customers from customer_basic
)
